In [1]:
# 全局设置
import os
import warnings
warnings.filterwarnings('ignore')
import datetime as dt

import numpy as np
import pandas as pd
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

# 因子库

以下以 HDF5DB 因子库作为示例，其他因子库类似操作

In [2]:
# 创建因子库
from QuantStudio.Factor.HDF5DB import HDF5DB

FDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()
print(qs_help(FDB))

类型: HDF5DB
模块: QuantStudio.Factor.HDF5DB
文档:
    基于 HDF5 文件的因子库
    每一张因子表对应一个文件夹, 其中的每个因子是文件夹下的一个 HDF5 文件
    每个 HDF5 文件有三个 Dataset: DateTime, ID, Data, 分别存储因子的时点序列, ID 序列, 因子值
    因子表的元数据存储在表文件夹下的特殊文件 _TableInfo.h5 中
    因子的元数据存储在 HDF5 文件的 attrs 中


In [3]:
# 因子库的参数集
display(Markdown(FDB.Args.info()))

* Name(名称): <class 'str'>, 默认值 HDF5DB
* MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录
* LockDir(锁目录): typing.Optional[typing.Annotated[pathlib.Path, PathType(path_type='dir')]], 默认值 None, 存放锁文件的目录, 默认 None 表示和主目录相同
* FileOpenRetryNum(文件打开重试次数): typing.Union[int, float], 默认值 inf, 打开数据文件错误时的重试次数
* ProcessLock(进程锁): <class 'bool'>, 默认值 True, 是否添加进程锁用于防止多进程间读写冲突

## 因子表列表

In [4]:
# 获取因子库中的因子表列表
print(FDB.TableNames)

['stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']


# 因子表

## 创建因子表

In [5]:
# 获取因子库中的某个因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
print(qs_help(FT))

类型: HDF5FactorTable
模块: QuantStudio.Factor.HDF5DB
文档:
    HDF5DB 库中因子表


In [6]:
# 因子表的参数集
display(Markdown(FT.Args.info()))

* Name(名称): <class 'str'>, 无默认值
* LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数
* OnlyStartLookBack(只起始日回溯): <class 'bool'>, 默认值 False, 如果为 True, 表示只对提取数据的第一个时点进行缺失填充, 之后的时点不填充
* OnlyLookBackNontarget(只回溯非目标日): <class 'bool'>, 默认值 False, 如果为 True, 表示只用不在提取时点序列中的数据进行缺失填充
* OnlyLookBackDT(只回溯时点): <class 'bool'>, 默认值 False, 如果为 True, 表示所有 ID 统一沿着时点字段进行回溯填充, 不单独填充
* TargetDT(目标时点): typing.Optional[datetime.datetime], 默认值 None, 非 None 表示只取该时点的值返回

## 因子列表

In [7]:
# 获取因子表中的因子列表
print(FT.FactorNames)

['amount', 'close', 'high', 'low', 'open', 'volume']


## 元信息

In [8]:
# 获取因子表元信息的方法
print(qs_help(FT.getMetaData))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getMetaData(key: Optional[str] = None) -> Union[Any, pandas.core.series.Series]
文档:
    获取因子表的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [9]:
# 获取因子表的所有元信息
print(FT.getMetaData())

Description    股票日K线
dtype: object


In [10]:
# 获取因子表的某个元信息
print(FT.getMetaData(key="Description"))

股票日K线


In [11]:
# 获取因子表中因子元信息的方法
print(qs_help(FT.getFactorMetaData))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getFactorMetaData(factor_names: Optional[List[str]] = None, key: Optional[str] = None) -> Union[pandas.core.frame.DataFrame, pandas.core.series.Series]
文档:
    获取因子的元信息, 元信息由若干个键值对组成
    
    Args:
        factor_names: 给定的因子名称列表, None 表示表中所有的因子
        key: 给定的元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key=None, 则返回 DataFrame(index=factor_names, columns=[所有的 key])
        如果 key 非 None 则返回该 key 对应的元信息, Series(index=factor_names)


In [12]:
# 获取某些因子的所有元信息
print(FT.getFactorMetaData(factor_names=["open", "close"], key=None))

      DataType
open    double
close   double


In [13]:
# 获取某些因子的某个元信息
print(FT.getFactorMetaData(factor_names=["open", "close"], key="DataType"))

open     double
close    double
dtype: object


## 时点序列

In [14]:
# 获取时点序列的方法
print(qs_help(FT.getDateTime))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getDateTime(ifactor_name: Optional[str] = None, iid: Optional[str] = None, start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
文档:
    获取时点序列
    
    Args:
        ifactor_name: 给定的因子名称, 非 None 表示获取该因子的时点序列, None 表示获取表的时点序列
        iid: 给定的 ID, 非 None 表示获取该 ID 的时点序列, None 表示获取所有的时点序列
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该因子表没有固定的时点序列或者无法获取


In [15]:
# 获取因子表的时点序列
DTs = FT.getDateTime()
print(DTs[0], " - ", DTs[-1])

2025-01-01 00:00:00  -  2025-04-10 00:00:00


In [16]:
# 给定起始时点和截止时点, 获取因子表的时点序列
FT.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[datetime.datetime(2025, 3, 1, 0, 0),
 datetime.datetime(2025, 3, 2, 0, 0),
 datetime.datetime(2025, 3, 3, 0, 0),
 datetime.datetime(2025, 3, 4, 0, 0),
 datetime.datetime(2025, 3, 5, 0, 0)]

In [17]:
# 给定起始时点, 截止时点, 目标因子, 获取因子表中指定因子的时点序列
FT.getDateTime(ifactor_name="close", start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[datetime.datetime(2025, 3, 1, 0, 0),
 datetime.datetime(2025, 3, 2, 0, 0),
 datetime.datetime(2025, 3, 3, 0, 0),
 datetime.datetime(2025, 3, 4, 0, 0),
 datetime.datetime(2025, 3, 5, 0, 0)]

In [18]:
# 给定起始时点, 截止时点, 目标因子, 目标 ID, 获取因子表中指定因子指定 ID 的时点序列
FT.getDateTime(ifactor_name="close", iid="000001.SZ", start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[Timestamp('2025-03-01 00:00:00'),
 Timestamp('2025-03-02 00:00:00'),
 Timestamp('2025-03-03 00:00:00'),
 Timestamp('2025-03-04 00:00:00'),
 Timestamp('2025-03-05 00:00:00')]

## ID 序列

In [19]:
# 获取因子表 ID 序列的方法
print(qs_help(FT.getID))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5FactorTable.getID(ifactor_name: Optional[str] = None, idt: Optional[datetime.datetime] = None) -> List[str]
文档:
    获取 ID 序列
    
    Args:
        ifactor_name: 给定的因子名称, 非 None 表示获取该因子的 ID 序列, None 表示获取表的 ID 序列
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该因子表没有固定的 ID 序列或者无法获取


In [20]:
# 获取因子表的 ID 序列
IDs = FT.getID()
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

000001.SZ , ...,  000020.SZ 共 20 个


In [21]:
# 给定目标因子, 获取因子表中指定因子的 ID 序列
IDs = FT.getID(ifactor_name="close")
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

000001.SZ , ...,  000020.SZ 共 20 个


In [22]:
# 给定目标因子, 目标时点, 获取因子表中指定因子指定时点的 ID 序列
IDs = FT.getID(ifactor_name="close", idt=dt.datetime(2025, 3, 1))
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

000001.SZ , ...,  000020.SZ 共 20 个


In [23]:
# 获取按条件筛选的 ID Mask
print(FT.getIDMask(idt=dt.datetime(2025, 3, 1), ids=None, id_filter_str="(@close<=40) & (@open>@close)"))

000001.SZ     True
000002.SZ     True
000003.SZ     True
000004.SZ    False
000005.SZ    False
000006.SZ    False
000007.SZ     True
000008.SZ     True
000009.SZ     True
000010.SZ     True
000011.SZ     True
000012.SZ     True
000013.SZ    False
000014.SZ    False
000015.SZ    False
000016.SZ    False
000017.SZ    False
000018.SZ    False
000019.SZ    False
000020.SZ    False
dtype: bool


In [24]:
# 获取按条件筛选的 ID 列表
IDs = FT.getFilteredID(idt=dt.datetime(2025, 3, 1), ids=None, id_filter_str="(@close<=40) & (@open>@close)")
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

000001.SZ , ...,  000012.SZ 共 9 个


## 读取数据

In [25]:
# 因子表数据读取方法
print(qs_help(FT.readData))

类型: method (bound to HDF5FactorTable)
模块: QuantStudio.Factor.FactorTable
签名: FactorTable.readData(factor_names: List[str], ids: List[str], dts: List[datetime.datetime], **kwargs) -> QuantStudio.Core.QSObject.Panel
文档:
    读取因子表数据
    
    Args:
        factor_names: 因子名称列表
        ids: ID 序列
        dts: 时点序列
    
    Returns:
        Panel(item=factor_names, major_axis=dts, minor_axis=ids)


In [26]:
# 给定因子列表, 时点列表, ID 列表, 获取数据
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))
Data = FT.readData(factor_names=["close", "open"], ids=["000001.SZ", "000002.SZ", "000058.SZ"], dts=DTs)
print("三维数据 : ")
print(Data)

iFactor = "close"
print(f"因子切片 : {iFactor}")
print(Data.loc[iFactor])

iDT = dt.datetime(2025, 3, 1)
print(f"时点切片 : {iDT}")
print(Data.loc[:, iDT])

iID = "000001.SZ"
print(f"ID 切片 : {iID}")
print(Data.loc[:, :, iID])

三维数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 3 (minor_axis)
Items axis: close to open
Major_axis axis: 2025-03-01 00:00:00 to 2025-03-05 00:00:00
Minor_axis axis: 000001.SZ to 000058.SZ
因子切片 : close
            000001.SZ  000002.SZ  000058.SZ
2025-03-01   1.360433   2.756653        NaN
2025-03-02   8.658053   3.880990        NaN
2025-03-03   0.355385   4.202593        NaN
2025-03-04   0.397833   0.138363        NaN
2025-03-05   8.640577   5.905727        NaN
时点切片 : 2025-03-01 00:00:00
              close      open
000001.SZ  1.360433  7.202659
000002.SZ  2.756653  9.253950
000058.SZ       NaN       NaN
ID 切片 : 000001.SZ
               close      open
2025-03-01  1.360433  7.202659
2025-03-02  8.658053  3.085280
2025-03-03  0.355385  9.829991
2025-03-04  0.397833  2.781165
2025-03-05  8.640577  2.593774


# 因子

## 获取因子

In [27]:
# 获取因子
F = FT.getFactor("close")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


In [28]:
# 因子的参数集
display(Markdown(F.Args.info()))

* Name(名称): <class 'str'>, 默认值 Factor
* Meta(元信息): <class 'dict'>, 默认值 {}
* SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None
* CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None
* CacheEnabled(启用缓存): <class 'bool'>, 默认值 True

## 元信息

In [29]:
# 获取因子元信息的方法
print(qs_help(F.getMetaData))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getMetaData(key: Optional[str] = None) -> Union[Any, pandas.core.series.Series]
文档:
    获取因子的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [30]:
# 获取因子的所有元信息
print(F.getMetaData(key=None))

DataType    double
dtype: object


In [31]:
# 获取因子的某个元信息
print(F.getMetaData(key="DataType"))

double


## 时点序列

In [32]:
# 获取因子时点序列的方法
print(qs_help(F.getDateTime))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getDateTime(iid: Optional[str] = None, start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None, **kwargs) -> List[datetime.datetime]
文档:
    获取时点序列
    
    Args:
        iid: 给定的 ID, 非 None 表示获取该 ID 的时点序列, None 表示获取所有的时点序列
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该因子没有固定的时点序列或者无法获取


In [33]:
# 获取因子的时点序列
DTs = F.getDateTime()
print(DTs[0], " - ", DTs[-1])

2025-01-01 00:00:00  -  2025-04-10 00:00:00


In [34]:
# 给定起始时点和截止时点, 获取因子的时点序列
F.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[datetime.datetime(2025, 3, 1, 0, 0),
 datetime.datetime(2025, 3, 2, 0, 0),
 datetime.datetime(2025, 3, 3, 0, 0),
 datetime.datetime(2025, 3, 4, 0, 0),
 datetime.datetime(2025, 3, 5, 0, 0)]

In [35]:
# 给定起始时点,截止时点,目标 ID, 获取因子指定 ID 的时点序列
F.getDateTime(iid="000001.SZ", start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))

[Timestamp('2025-03-01 00:00:00'),
 Timestamp('2025-03-02 00:00:00'),
 Timestamp('2025-03-03 00:00:00'),
 Timestamp('2025-03-04 00:00:00'),
 Timestamp('2025-03-05 00:00:00')]

## ID 序列

In [36]:
# 获取因子 ID 序列的方法
print(qs_help(F.getID))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getID(idt: Optional[datetime.datetime] = None, **kwargs) -> List[str]
文档:
    获取 ID 序列
    
    Args:
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该因子没有固定的 ID 序列或者无法获取


In [37]:
# 获取因子的 ID 序列
IDs = F.getID()
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


In [38]:
# 给定目标时点, 获取因子指定时点的 ID 序列
IDs = F.getID(idt=dt.datetime(2025, 3, 1))
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


## 读取数据

In [39]:
# 读取因子数据的方法
print(qs_help(F.readData))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.readData(ids: List[str], dts: List[datetime.datetime], **kwargs) -> pandas.core.frame.DataFrame
文档:
    读取因子数据
    
    Args:
        ids: ID 序列
        dts: 时点序列
        kwargs: 可传入的参数有
            dt_ruler: 时点标尺序列, 如果没有传入则为 dts
            section_ids: 截面 ID 序列，如果没有传入则为因子参数集中指定的 SectionIDs, 如果参数集中未指定则为 ids
    
    Returns:
        DataFrame(index=dts, columns=ids)


In [40]:
# 给定时点列表, ID 列表, 获取数据
DTs = F.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5))
print("二维数据 : ")
print(F.readData(ids=["000001.SZ", "000002.SZ", "000058.SZ"], dts=DTs))

二维数据 : 
            000001.SZ  000002.SZ  000058.SZ
2025-03-01   1.360433   2.756653        NaN
2025-03-02   8.658053   3.880990        NaN
2025-03-03   0.355385   4.202593        NaN
2025-03-04   0.397833   0.138363        NaN
2025-03-05   8.640577   5.905727        NaN


# 数据写入

只有支持写入的因子库（继承自 QuantStudio.Factor.FactorDB.WritableFactorDB）才能使用以下方法

In [41]:
# 因子库数据写入方法
print(qs_help(FDB.writeData))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.writeData(data: QuantStudio.Core.QSObject.Panel, table_name: str, if_exists: Literal['update', 'replace', 'append'] = 'update', data_type: Dict[str, Literal['double', 'string', 'object']] = {}, **kwargs)
文档:
    写入数据
    
    Args:
        data: 待写入的因子数据, Panel(items=[因子], major_axis=[时点], minor_axis=[ID])
        table_name: 因子表名称
        if_exists: 如果该因子已经存在时数据写入的方式, update 表示用新数据更新原数据, append 表示不更新原数据而只增加原来没有的数据, replace 表示完全用新数据替换原数据, 等同于先删除原数据再写入
        data_type: 待写入因子的数据类型, {因子名称: "double" or "string" or "object"}, 如果 data_type 未指定某个因子的数据类型，则交由系统判定


In [42]:
# 数据写入
from QuantStudio.Core.QSObject import Panel

IDs = [str(i).zfill(6) + ".SZ" for i in range(1, 4)]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]

Data = Panel({
    "Factor1": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs),
    "Factor2": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs)
})
print("待写入的数据 : ")
print(Data)

TargetTable = "TestTable"
FDB.writeData(data=Data, table_name=TargetTable, if_exists="update", data_type={"Factor1":"double", "Factor2":"double"})
print("写入后的因子表 : ")
print(FDB.TableNames)
print("写入的因子 : ")
print(FDB.getTable(TargetTable).FactorNames)

待写入的数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 3 (minor_axis)
Items axis: Factor1 to Factor2
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000003.SZ
写入后的因子表 : 
['TestTable', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
写入的因子 : 
['Factor1', 'Factor2']


# 因子库其他操作

## 设置因子的元信息

In [43]:
# 设置因子元信息的方法
print(qs_help(FDB.setFactorMetaData))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.setFactorMetaData(table_name: str, ifactor_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
文档:
    设置因子的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 因子表名称
        ifactor_name: 因子名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [44]:
# 设置因子的元信息
TargetTable = "TestTable"
TargetFactor = "Factor1"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))
FDB.setFactorMetaData(table_name=TargetTable, ifactor_name=TargetFactor, key="Description", value="这是一个测试因子")
print("设置后的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

设置前的元信息 : 
        DataType
Factor1   double
设置后的元信息 : 
        DataType Description
Factor1   double    这是一个测试因子


## 重命名因子

In [45]:
# 重命名因子的方法
print(qs_help(FDB.renameFactor))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.renameFactor(table_name: str, old_factor_name: str, new_factor_name: str)
文档:
    对给定表中的因子重命名
    
    Args:
        table_name: 因子表名称
        old_factor_name: 原因子名
        new_factor_name: 新因子名


In [46]:
# 重命名因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("重命名前因子列表 : ")
print(FT.FactorNames)
FDB.renameFactor(table_name=TargetTable, old_factor_name="Factor1", new_factor_name="NewFactor1")
print("重命名后因子列表 : ")
print(FT.FactorNames)

重命名前因子列表 : 
['Factor1', 'Factor2']
重命名后因子列表 : 
['Factor2', 'NewFactor1']


## 删除因子

In [47]:
# 删除因子的方法
print(qs_help(FDB.deleteFactor))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.deleteFactor(table_name: str, factor_names: List[str])
文档:
    删除给定表中的某些因子
    
    Args:
        table_name: 因子表名称
        factor_names: 待删除的因子名列表


In [48]:
# 删除因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("删除前因子列表 : ")
print(FT.FactorNames)
FDB.deleteFactor(table_name=TargetTable, factor_names=["NewFactor1"])
print("删除后因子列表 : ")
print(FT.FactorNames)

删除前因子列表 : 
['Factor2', 'NewFactor1']
删除后因子列表 : 
['Factor2']


## 设置表的元信息

In [49]:
# 设置因子表元信息的方法
print(qs_help(FDB.setTableMetaData))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.setTableMetaData(table_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
文档:
    设置因子表的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 因子表名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [50]:
# 设置表的元信息
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getMetaData())
FDB.setTableMetaData(table_name=TargetTable, key="Description", value="这是一张测试表")
print("设置后的元信息 : ")
print(FT.getMetaData())

设置前的元信息 : 
Series([], dtype: object)
设置后的元信息 : 
Description    这是一张测试表
dtype: object


In [51]:
# 设置表的元信息
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getMetaData())
FDB.setTableMetaData(table_name=TargetTable, meta_data={"Description": "这还是一张测试表", "aha": 123})
print("设置后的元信息 : ")
print(FT.getMetaData())

设置前的元信息 : 
Description    这是一张测试表
dtype: object
设置后的元信息 : 
Description    这还是一张测试表
aha                 123
dtype: object


## 重命名表

In [52]:
# 重命名表的方法
print(qs_help(FDB.renameTable))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.renameTable(old_table_name: str, new_table_name: str)
文档:
    重命名表
    
    Args:
        old_table_name: 原表名
        new_table_name: 新表名


In [53]:
# 重命名表
print("重命名前因子表 : ")
print(FDB.TableNames)
FDB.renameTable(old_table_name="TestTable", new_table_name="NewTestTable")
print("重命名后因子表 : ")
print(FDB.TableNames)

重命名前因子表 : 
['TestTable', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
重命名后因子表 : 
['NewTestTable', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']


## 删除表

In [54]:
# 删除表的方法
print(qs_help(FDB.deleteTable))

类型: method (bound to HDF5DB)
模块: QuantStudio.Factor.HDF5DB
签名: HDF5DB.deleteTable(table_name: str)
文档:
    删除表
    
    Args:
        table_name: 表名


In [55]:
# 删除表
print("删除前因子表 : ")
print(FDB.TableNames)
FDB.deleteTable(table_name="NewTestTable")
print("删除后因子表 : ")
print(FDB.TableNames)

删除前因子表 : 
['NewTestTable', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
删除后因子表 : 
['stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
